In [3]:
import os, pandas as pd, h5py

h5_file_path = r"E:\bilat_asymmetry_analysis\raw\TeLC_Silencing\WA021\20260401\Post_Day10_2.0\WA021_L_MRN_TelC_04_TelC_Day10_notail_2.0_2026-04-01_0001.h5"
print("exists:", os.path.exists(h5_file_path))
print("size:", os.path.getsize(h5_file_path) if os.path.exists(h5_file_path) else "n/a")

try:
    with pd.HDFStore(h5_file_path, "r") as s:
        print("pandas keys:", s.keys())
except Exception as e:
    print("HDFStore open failed:", type(e).__name__, e)

try:
    with h5py.File(h5_file_path, "r") as f:
        print("h5py top-level keys:", list(f.keys()))
except Exception as e:
    print("h5py open failed:", type(e).__name__, e)

exists: True
size: 129861424
pandas keys: []
h5py top-level keys: ['header', 'sweep_0001']


Convert Pulse into TTLs

In [6]:
import h5py
import numpy as np
import pandas as pd
from pathlib import Path

#h5_file = Path(r"D:\ear_movement_tail_pinch_20260326\tail_pinch_WA024_L_R_tail_pinch_ear_tracking_2026-03-28_0001.h5")
#h5_file = Path(r"E:\whisker_asymmetry\nob_frame_videos_neuralyzer\Opto_TelC_ttls\WA029_0_MRN_TelC6_02_Day7_2026-05-15_001_0001.h5")

h5_file = Path(r"E:\bilat_asymmetry_analysis\raw\TeLC_Silencing\WA021\20260401\Post_Day10_2.0\WA021_L_MRN_TelC_04_TelC_Day10_notail_2.0_2026-04-01_0001.h5")
out_csv = h5_file.with_name(h5_file.stem + "_pulsepal_ttls.csv")


In [7]:
# PulsePal TTL input line (DI1 -> bit 1 in your setup)
PULSEPAL_BIT = 1

with h5py.File(h5_file, "r") as f:
    fs = float(np.array(f["/header/AcquisitionSampleRate"]).reshape(-1)[0])
    digital = np.array(f["/sweep_0001/digitalScans"])[0].astype(np.uint16)

# Estimate camera FPS from camera TTL on DI0
cam = ((digital & (1 << 0)) != 0)
cam_rise = np.flatnonzero((~cam[:-1]) & cam[1:]) + 1
if cam_rise.size < 2:
    raise RuntimeError("Not enough camera rising edges to estimate FPS.")
fps = 1.0 / np.median(np.diff(cam_rise) / fs)
print("Estimated camera FPS:", fps)

# PulsePal TTL decode
ttl = ((digital & (1 << PULSEPAL_BIT)) != 0)

# Rising/falling edges in DAQ samples
rise_idx = np.flatnonzero((~ttl[:-1]) & ttl[1:]) + 1
fall_idx = np.flatnonzero(ttl[:-1] & (~ttl[1:])) + 1

# Pair each rise with next fall
j = np.searchsorted(fall_idx, rise_idx, side="right")
ok = j < fall_idx.size
rise_idx = rise_idx[ok]
fall_idx = fall_idx[j[ok]]
ok2 = fall_idx > rise_idx
rise_idx = rise_idx[ok2]
fall_idx = fall_idx[ok2]

# Convert sample indices to frame indices using estimated FPS
samples_per_frame = fs / fps
rise_frame = np.rint(rise_idx / samples_per_frame).astype(int)
fall_frame = np.rint(fall_idx / samples_per_frame).astype(int)
duration_frames = (fall_frame - rise_frame).astype(int)

df = pd.DataFrame({
    "pulse_index": np.arange(len(rise_idx), dtype=int),
    "rise_sample": rise_idx.astype(int),
    "fall_sample": fall_idx.astype(int),
    "rise_time_s": rise_idx / fs,
    "fall_time_s": fall_idx / fs,
    "duration_s": (fall_idx - rise_idx) / fs,
    "rise_frame": rise_frame,
    "fall_frame": fall_frame,
    "duration_frames": duration_frames,
    "estimated_fps": fps
})

df.to_csv(out_csv, index=False)
print(f"Saved {len(df)} pulses to: {out_csv}")

Estimated camera FPS: 500.0
Saved 65 pulses to: E:\bilat_asymmetry_analysis\raw\TeLC_Silencing\WA021\20260401\Post_Day10_2.0\WA021_L_MRN_TelC_04_TelC_Day10_notail_2.0_2026-04-01_0001_pulsepal_ttls.csv


Align H5 channels to MP4 frames

This cell treats camera TTL rising edges on `CAMERA_BIT` as the video frame clock. H5 samples before the first camera edge and after the last camera edge are treated as pre-roll and post-roll. Set `VIDEO_START_CAMERA_EDGE` to a different edge only if camera TTLs are present before the MP4 starts.

In [9]:
mp4_candidates = sorted(h5_file.parent.glob("*.mp4"))
if len(mp4_candidates) != 1:
    raise RuntimeError(
        f"Expected exactly one MP4 beside the H5; found {len(mp4_candidates)}. "
        "Set mp4_file explicitly."
    )
mp4_file = mp4_candidates[0]
print(f"Using MP4: {mp4_file}")

Using MP4: E:\bilat_asymmetry_analysis\raw\TeLC_Silencing\WA021\20260401\Post_Day10_2.0\WA021_L_MRN_TelC_04_TelC_Day10_notail_2_noB_rot180.mp4


In [12]:
import cv2
import h5py
import numpy as np
import pandas as pd

CAMERA_BIT = 0
PULSEPAL_BIT = 1
VIDEO_START_CAMERA_EDGE = 0

with h5py.File(h5_file, "r") as h5:
    fs = float(np.asarray(
        h5["/header/AcquisitionSampleRate"]
    ).reshape(-1)[0])
    digital = np.asarray(
        h5["/sweep_0001/digitalScans"]
    )[0].astype(np.uint16)

camera_signal = (digital & (1 << CAMERA_BIT)) != 0
camera_rise = np.flatnonzero(
    (~camera_signal[:-1]) & camera_signal[1:]
) + 1

if camera_rise.size < 2:
    raise RuntimeError("Not enough camera TTL edges to estimate FPS.")

fps = fs / np.median(np.diff(camera_rise))
video_start_sample = camera_rise[VIDEO_START_CAMERA_EDGE]

cap = cv2.VideoCapture(str(mp4_file))
video_frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
video_header_fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()

if video_frame_count <= 0:
    raise RuntimeError("Could not read the MP4 frame count.")

if not 0 <= VIDEO_START_CAMERA_EDGE < camera_rise.size:
    raise ValueError("VIDEO_START_CAMERA_EDGE is outside the camera TTL range.")

pulse_signal = (digital & (1 << PULSEPAL_BIT)) != 0
pulse_rise = np.flatnonzero(
    (~pulse_signal[:-1]) & pulse_signal[1:]
) + 1
pulse_fall = np.flatnonzero(
    pulse_signal[:-1] & (~pulse_signal[1:])
) + 1

fall_position = np.searchsorted(
    pulse_fall,
    pulse_rise,
    side="right"
)
valid_pairs = fall_position < pulse_fall.size
pulse_rise = pulse_rise[valid_pairs]
pulse_fall = pulse_fall[fall_position[valid_pairs]]
valid_pairs = pulse_fall > pulse_rise
pulse_rise = pulse_rise[valid_pairs]
pulse_fall = pulse_fall[valid_pairs]

def samples_to_video_frames(samples):
    # Use elapsed H5 time, not camera-edge counts, because TTL edges can drop.
    elapsed_samples = samples - video_start_sample
    return np.rint(elapsed_samples * fps / fs).astype(int)

rise_frame = samples_to_video_frames(pulse_rise)
fall_frame = samples_to_video_frames(pulse_fall)

alignment = pd.DataFrame({
    "pulse_index": np.arange(pulse_rise.size, dtype=int),
    "rise_sample": pulse_rise.astype(int),
    "fall_sample": pulse_fall.astype(int),
    "rise_h5_time_s": pulse_rise / fs,
    "fall_h5_time_s": pulse_fall / fs,
    "rise_video_frame": rise_frame.astype(int),
    "fall_video_frame": fall_frame.astype(int),
    "rise_video_time_s": rise_frame / fps,
    "fall_video_time_s": fall_frame / fps,
    "duration_video_frames": (fall_frame - rise_frame).astype(int),
    "estimated_fps": fps,
})

alignment_csv = h5_file.with_name(
    h5_file.stem + "_h5_to_mp4_alignment_time_based.csv"
)
alignment.to_csv(alignment_csv, index=False)

print(f"Estimated camera FPS: {fps:.6f}")
print(f"H5 pre-roll before first camera TTL: {camera_rise[0] / fs:.3f} s")
print(f"H5 post-roll after last camera TTL: {(len(digital) - camera_rise[-1]) / fs:.3f} s")
print(f"Camera TTL edges: {camera_rise.size}")
print(f"MP4 header frame count: {video_frame_count}")
print(f"MP4 header FPS (often unreliable): {video_header_fps}")
print(f"Video start camera edge: {VIDEO_START_CAMERA_EDGE}")
print(f"Saved alignment table to: {alignment_csv}")

if camera_rise.size != video_frame_count:
    print(
        "WARNING: camera-edge count differs from MP4 frame count. "
        "Time-based mapping assumes a stable camera clock and no video frame drops."
    )

invalid_frames = (
    (rise_frame < 0) | (rise_frame >= video_frame_count)
)
print(f"Pulse rises outside MP4 frame range: {invalid_frames.sum()}")

Estimated camera FPS: 500.000000
H5 pre-roll before first camera TTL: 3.387 s
H5 post-roll after last camera TTL: 1.500 s
Camera TTL edges: 285469
MP4 header frame count: 283580
MP4 header FPS (often unreliable): 30.0
Video start camera edge: 0
Saved alignment table to: E:\bilat_asymmetry_analysis\raw\TeLC_Silencing\WA021\20260401\Post_Day10_2.0\WA021_L_MRN_TelC_04_TelC_Day10_notail_2.0_2026-04-01_0001_h5_to_mp4_alignment_time_based.csv
Pulse rises outside MP4 frame range: 0


Export aligned camera and PulsePal events

The exported `video_time_s` and `video_frame` columns use the first camera TTL as video time zero. `h5_time_s` remains the original H5 acquisition time. No manual-contact correction is applied because the contact is a separate, manually observed event.

In [ ]:
def make_event_rows(channel_name, channel_number, signal):
    rising = np.flatnonzero((~signal[:-1]) & signal[1:]) + 1
    falling = np.flatnonzero(signal[:-1] & (~signal[1:])) + 1
    samples = np.concatenate([rising, falling])
    edges = np.concatenate([
        np.full(rising.size, "rising", dtype=object),
        np.full(falling.size, "falling", dtype=object),
    ])
    order = np.argsort(samples)
    samples = samples[order]
    edges = edges[order]
    video_time = (samples - video_start_sample) / fs
    video_frame = np.rint(video_time * fps).astype(int)
    return pd.DataFrame({
        "feature": channel_name,
        "channel": channel_number,
        "edge": edges,
        "h5_sample": samples.astype(int),
        "h5_time_s": samples / fs,
        "video_time_s": video_time,
        "video_frame": video_frame,
        "ttl_value": np.where(edges == "rising", 1, 0),
    })

camera_events = make_event_rows(
    "camera_ttl",
    CAMERA_BIT,
    camera_signal,
)
pulsepal_events = make_event_rows(
    "pulsepal_ttl",
    PULSEPAL_BIT,
    pulse_signal,
)
events = pd.concat(
    [camera_events, pulsepal_events],
    ignore_index=True,
).sort_values("h5_sample")

events_csv = h5_file.with_name(
    h5_file.stem + "_camera_pulsepal_events_aligned.csv"
)
events.to_csv(events_csv, index=False)

pulsepal_intervals = pd.DataFrame({
    "feature": "pulsepal_interval",
    "channel": PULSEPAL_BIT,
    "start_sample": pulse_rise.astype(int),
    "end_sample": pulse_fall.astype(int),
    "start_h5_time_s": pulse_rise / fs,
    "end_h5_time_s": pulse_fall / fs,
    "start_time_s": (pulse_rise - video_start_sample) / fs,
    "end_time_s": (pulse_fall - video_start_sample) / fs,
    "start_frame": samples_to_video_frames(pulse_rise),
    "end_frame": samples_to_video_frames(pulse_fall),
    "duration_s": (pulse_fall - pulse_rise) / fs,
})
intervals_csv = h5_file.with_name(
    h5_file.stem + "_pulsepal_intervals_aligned.csv"
)
pulsepal_intervals.to_csv(intervals_csv, index=False)

print(f"Saved edge events to: {events_csv}")
print(f"Saved PulsePal intervals to: {intervals_csv}")
print(f"Camera events: {len(camera_events)}")
print(f"PulsePal events: {len(pulsepal_events)}")
print(f"PulsePal intervals: {len(pulsepal_intervals)}")